DAY  13-april oupby() + Aggregations + value_counts() + unique() + nunique()


In [43]:
import pandas as pd
import numpy as np
df = pd.read_csv('/content/Sample - Superstore.csv', encoding='latin-1')
# encoding = latin1 -- in this file special characters

print(df.shape)
print(df.columns.tolist())
print(df.head(3))



(9994, 21)
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']
   Row ID        Order ID Order Date   Ship Date     Ship Mode Customer ID  \
0       1  CA-2016-152156  11/8/2016  11/11/2016  Second Class    CG-12520   
1       2  CA-2016-152156  11/8/2016  11/11/2016  Second Class    CG-12520   
2       3  CA-2016-138688  6/12/2016   6/16/2016  Second Class    DV-13045   

     Customer Name    Segment        Country         City  ... Postal Code  \
0      Claire Gute   Consumer  United States    Henderson  ...       42420   
1      Claire Gute   Consumer  United States    Henderson  ...       42420   
2  Darrin Van Huff  Corporate  United States  Los Angeles  ...       90036   

   Region       Product ID         Category Sub-Category  \
0   South  FUR-BO-10001798        Furniture

In [44]:
# Quick overview
print(df.dtypes)
print(df.isnull().sum())   # missing values check
print(df.describe())

Row ID             int64
Order ID          object
Order Date        object
Ship Date         object
Ship Mode         object
Customer ID       object
Customer Name     object
Segment           object
Country           object
City              object
State             object
Postal Code        int64
Region            object
Product ID        object
Category          object
Sub-Category      object
Product Name      object
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64
            Row ID   Postal Code         Sales   

In [45]:
# groupby - data divide into groups  , then after calculation using group
# 3 steps
# split - divide data into groups
# 2- apply - implement function to every group sum mean
# 3 - combien - results at one table
"""
Original Data:           Groups:              Result:
Region  Sales            West:  [100,200,50]  West   350
West    100      →       East:  [300,150]  →  East   450
East    300              South: [80,120]       South  200
West    200
East    150
South   80
West    50
South   120"""

'\nOriginal Data:           Groups:              Result:\nRegion  Sales            West:  [100,200,50]  West   350\nWest    100      →       East:  [300,150]  →  East   450\nEast    300              South: [80,120]       South  200\nWest    200\nEast    150\nSouth   80\nWest    50\nSouth   120'

In [46]:
# 6.2 - single column groupby() + aggregations
# aggregation - single value return ex - sum , mean , max , median

# region wise total sales
region_sales = df.groupby('Region')['Sales'].sum()
print(region_sales.sort_values(ascending=False))
print(type(region_sales)) # series

Region
West       725457.8245
East       678781.2400
Central    501239.8908
South      391721.9050
Name: Sales, dtype: float64
<class 'pandas.core.series.Series'>


In [47]:
# category - wise average profit
cat_profit = df.groupby('Category')['Profit'].mean().round(2)
print(cat_profit)

Category
Furniture           8.70
Office Supplies    20.33
Technology         78.75
Name: Profit, dtype: float64


In [48]:
# all aggregations

# count
df.groupby('Region')['Sales'].count()

df.groupby('Region')['Sales'].mean()

df.groupby('Region')['Sales'].median()

df.groupby('Region')['Sales'].max()

df.groupby('Region')['Sales'].min()

df.groupby('Region')['Sales'].std()

# std - spread data out mean around how much data is spread out


,Sales
Region,
Central,632.779010
East,620.712652
South,774.796273
West,524.876877


In [49]:
# 6.3 agg() : multiple aggregations at one time

# regions basis sum ,mean median , count
result = df.groupby('Region')['Sales'].agg(['sum','mean','count','std','max','min'])
print(result.round(2))

               sum    mean  count     std       max   min
Region                                                   
Central  501239.89  215.77   2323  632.78  17499.95  0.44
East     678781.24  238.34   2848  620.71  11199.97  0.85
South    391721.90  241.80   1620  774.80  22638.48  1.17
West     725457.82  226.49   3203  524.88  13999.96  0.99


In [50]:
result = df.groupby('Category').agg({
    'Sales':    ['sum', 'mean', 'max'],
    'Profit':   ['sum', 'mean'],
    'Quantity': ['sum', 'mean']
}).round(2)

print(result)

                     Sales                       Profit        Quantity      
                       sum    mean       max        sum   mean      sum  mean
Category                                                                     
Furniture        741999.80  349.83   4416.17   18451.27   8.70     8028  3.79
Office Supplies  719047.03  119.32   9892.74  122490.80  20.33    22906  3.80
Technology       836154.03  452.71  22638.48  145454.95  78.75     6939  3.76


In [51]:
# custom columns names
result = df.groupby('Region').agg(
    Total_Sales   = ('Sales',  'sum'),
    Avg_Sales     = ('Sales',  'mean'),
    Total_Profit  = ('Profit', 'sum'),
    Order_Count   = ('Order ID', 'count')
).round(2)

print(result)

         Total_Sales  Avg_Sales  Total_Profit  Order_Count
Region                                                    
Central    501239.89     215.77      39706.36         2323
East       678781.24     238.34      91522.78         2848
South      391721.90     241.80      46749.43         1620
West       725457.82     226.49     108418.45         3203


In [52]:
# 6.4 multiple columns to groupby()
# region + category basis groupby
result  = df.groupby(['Region','Category'])['Sales'].sum().round(4)
print(result)

Region   Category       
Central  Furniture          163797.1638
         Office Supplies    167026.4150
         Technology         170416.3120
East     Furniture          208291.2040
         Office Supplies    205516.0550
         Technology         264973.9810
South    Furniture          117298.6840
         Office Supplies    125651.3130
         Technology         148771.9080
West     Furniture          252612.7435
         Office Supplies    220853.2490
         Technology         251991.8320
Name: Sales, dtype: float64


In [53]:
# unstack
# convert inner index to columns - like pivot table
result_unstacked = df.groupby(['Region','Category'])['Sales'].sum().unstack()
print(result_unstacked.round(2))

Category  Furniture  Office Supplies  Technology
Region                                          
Central   163797.16        167026.42   170416.31
East      208291.20        205516.06   264973.98
South     117298.68        125651.31   148771.91
West      252612.74        220853.25   251991.83


In [54]:
# every row total sales
df['Region_Total_Sales'] = df.groupby('Region')['Sales'].transform('sum')

# evry row category avg profit
df['Category_Avg_Profit'] = df.groupby('Category')['Profit'].transform('mean')


df['Sales_Contribution_%'] = (df['Sales'] / df['Region_Total_Sales'] * 100).round(4)

print(df[['Region', 'Sales', 'Region_Total_Sales', 'Sales_Contribution_%']].head(8))

  Region     Sales  Region_Total_Sales  Sales_Contribution_%
0  South  261.9600         391721.9050                0.0669
1  South  731.9400         391721.9050                0.1869
2   West   14.6200         725457.8245                0.0020
3  South  957.5775         391721.9050                0.2445
4  South   22.3680         391721.9050                0.0057
5   West   48.8600         725457.8245                0.0067
6   West    7.2800         725457.8245                0.0010
7   West  907.1520         725457.8245                0.1250


In [55]:
# transorm and agg diffrence
# agg() -- summary table compressed
df.groupby('Region')['Sales'].agg('sum')

# result - 4 rows ( one per region )

# transform - original shape maintain
df.groupby('Region')['Sales'].transform('sum')

# at every row that region total - all rows

,Sales
0,391721.9050
1,391721.9050
2,725457.8245
3,391721.9050
4,391721.9050
...,...
9989,391721.9050
9990,725457.8245
9991,725457.8245
9992,725457.8245


In [56]:
# 6.7 value_counts - frequency count
# how many times ship mode
print(df['Ship Mode'].value_counts())

Ship Mode
Standard Class    5968
Second Class      1945
First Class       1538
Same Day           543
Name: count, dtype: int64


In [57]:
# see in percentage
print(df['Ship Mode'].value_counts(normalize=True).round(3) * 100)

Ship Mode
Standard Class    59.7
Second Class      19.5
First Class       15.4
Same Day           5.4
Name: proportion, dtype: float64


In [66]:
# 6.8 unique and nunique
# nunique - n + unique values return
# how many unique states
print(df['State'].nunique()) # 49
#print(df['State'].unique()) # columns return

# unique customers
df['Customer Name'].nunique()

df['Product Name'].nunique()
# unique products

# at every column unique value count
print(df.nunique())

49
Row ID                  9994
Order ID                5009
Order Date              1237
Ship Date               1334
Ship Mode                  4
Customer ID              793
Customer Name            793
Segment                    3
Country                    1
City                     531
State                     49
Postal Code              631
Region                     4
Product ID              1862
Category                   3
Sub-Category              17
Product Name            1850
Sales                   5825
Quantity                  14
Discount                  12
Profit                  7287
Region_Total_Sales         4
Category_Avg_Profit        3
Sales_Contribution_%    1622
dtype: int64


In [68]:
# at groupby  unique
unique_customers = df.groupby('Region')['Customer Name'].nunique()
print(unique_customers)

Region
Central    629
East       674
South      512
West       686
Name: Customer Name, dtype: int64


In [82]:
# 6.9 Real world Business Analysis - Superstore
print("       SUPERSTORE BUSINESS ANALYSIS REPORT")

# overall summary
print(f"\n Total Orders:   {df['Order ID'].nunique():,}")
print(f"Total Customers: {df['Customer Name'].nunique():,}")
print(f"Total Products:  {df['Product Name'].nunique():,}")
print(f"Total Sales:    ₹{df['Sales'].sum():,.2f}")
print(f"Total Profit:   ₹{df['Profit'].sum():,.2f}")
print(f"Profit Margin:   {(df['Profit'].sum()/df['Sales'].sum()*100):.2f}%")



# 2 - region performance
region_perf = df.groupby('Region').agg(
    Sales = ('Sales','sum'),
    Profit = ('Profit','sum'),
    Orders      = ('Order ID', 'nunique'),
    Customers   = ('Customer Name', 'nunique')
).round(2)
region_perf['Profit_Margin%'] = (
    region_perf['Profit'] / region_perf['Sales'] * 100
).round(2)
print(region_perf.sort_values('Sales', ascending=False))


# category analysis
print("\n--- Category-wise Analysis ---")
cat_analysis = df.groupby('Category').agg(
    Total_Sales   = ('Sales', 'sum'),
    Total_Profit  = ('Profit', 'sum'),
    Avg_Discount  = ('Discount', 'mean'),
    Product_Count = ('Product Name', 'nunique')
).round(2)
cat_analysis['Margin%'] = (
    cat_analysis['Total_Profit'] / cat_analysis['Total_Sales'] * 100
).round(2)
print(cat_analysis)


# 4- losing money

print("\n--- Sub-Categories LOSING Money ---")
subcat_profit = df.groupby('Sub-Category')['Profit'].sum().round(2)
losing = subcat_profit[subcat_profit < 0].sort_values()
print(losing)

# ── 5. Best & Worst Products ──
print("\n--- Top 5 Most Profitable Products ---")
top_products = df.groupby('Product Name')['Profit'].sum()
print(top_products.nlargest(5).round(2))

print("\n--- Top 5 Least Profitable Products ---")
print(top_products.nsmallest(5).round(2))

# ── 6. Ship Mode Usage ──
print("\n--- Shipping Mode Distribution ---")
print(df['Ship Mode'].value_counts())
print(df['Ship Mode'].value_counts(normalize=True).round(3) * 100)

# ── 7. Customer Segment Analysis ──
print("\n--- Customer Segment Performance ---")
seg = df.groupby('Segment').agg(
    Sales  = ('Sales', 'sum'),
    Profit = ('Profit', 'sum'),
    Orders = ('Order ID', 'nunique')
).round(2)
print(seg)



       SUPERSTORE BUSINESS ANALYSIS REPORT

 Total Orders:   5,009
Total Customers: 793
Total Products:  1,850
Total Sales:    ₹2,297,200.86
Total Profit:   ₹286,397.02
Profit Margin:   12.47%
             Sales     Profit  Orders  Customers  Profit_Margin%
Region                                                          
West     725457.82  108418.45    1611        686           14.94
East     678781.24   91522.78    1401        674           13.48
Central  501239.89   39706.36    1175        629            7.92
South    391721.90   46749.43     822        512           11.93

--- Category-wise Analysis ---
                 Total_Sales  Total_Profit  Avg_Discount  Product_Count  \
Category                                                                  
Furniture          741999.80      18451.27          0.17            380   
Office Supplies    719047.03     122490.80          0.16           1058   
Technology         836154.03     145454.95          0.13            412   

         